In [ ]:
Task 1: The Tiny Fingerprint (Linear Scale)To manually scale an image by 300% on both axes, a $2\times2$ scaling matrix is required, structured as:$$M_{scale} = \begin{bmatrix} s_x & 0 \\ 0 & s_y \end{bmatrix}$$
To keep the fingerprint perfectly centered after it grows, you must calculate the difference between the original center and the scaled center, requiring an upgrade to a $2\times3$ affine matrix to apply the translation.   Pythonimport cv2
import numpy as np

image = cv2.imread('fingerprint.jpg')
height, width = image.shape[:2]

# Enlarge by 300% on both axes
sx, sy = 3.0, 3.0

# Calculate translation to keep the fingerprint perfectly centered
# Offset = Original Center - Scaled Center
tx = (width / 2) - ((width / 2) * sx)
ty = (height / 2) - ((height / 2) * sy)

# Upgrade to 2x3 matrix to allow for the centering movement
M = np.float32([
    [sx, 0, tx],
    [0, sy, ty]
])

scaled_fingerprint = cv2.warpAffine(image, M, (int(width * sx), int(height * sy)))
Task 2: The Dizzy Satellite (Linear Rotation)A manual rotation matrix utilizes sine and cosine to balance both axes. To prevent the corners of the city from being chopped off, the new bounding box dimensions must be computed mathematically using the absolute values of the trigonometric functions.   Pythonimport cv2
import numpy as np
import math

image = cv2.imread('city.jpg')
h, w = image.shape[:2]

# Formulate 2x2 rotation matrix using sine and cosine for 45 degrees[cite: 64]
angle_rad = math.radians(45)
cos_a = math.cos(angle_rad)
sin_a = math.sin(angle_rad)

# Calculate new image dimensions to prevent chopping corners[cite: 64]
new_w = int((h * abs(sin_a)) + (w * abs(cos_a)))
new_h = int((h * abs(cos_a)) + (w * abs(sin_a)))

# Calculate translation to shift the image into the center of the new bounding box
tx = (new_w / 2) - (w / 2)
ty = (new_h / 2) - (h / 2)

# Build the 2x3 matrix (OpenCV coordinate system inverts the sin signs for standard rotation)
M_rot = np.float32([
    [cos_a, sin_a, tx],
    [-sin_a, cos_a, ty]
])

rotated_city = cv2.warpAffine(image, M_rot, (new_w, new_h))
Task 3: The Fast Train Barcode (Linear Shear)To fix a rightward slant, a $2\times2$ shear matrix pushes the pixels horizontally using a constant factor $k_x$[cite: 64].Pythonimport cv2
import numpy as np

image = cv2.imread('barcode.jpg')
height, width = image.shape[:2]

# Shear factor to push pixels horizontally and fix the slant[cite: 64]
kx = -0.4 # Negative value pushes pixels to the left

# Construct a 2x3 shear matrix[cite: 64]
M_shear = np.float32([
    [1, kx, 0],
    [0, 1, 0]
])

# Expand the width to accommodate the pixel shift
new_width = int(width + (abs(kx) * height))
fixed_barcode = cv2.warpAffine(image, M_shear, (new_width, height))
Task 4: The Hidden Treasure Map (Affine Translation)To translate an image, the $2\times2$ matrix must be upgraded to a $3\times3$ (or $2\times3$ for OpenCV warpAffine) to fold the translation values into the multiplication[cite: 64]. In image coordinates, moving "up" requires a negative $y$ value.   Pythonimport cv2
import numpy as np

image = cv2.imread('map.jpg')
height, width = image.shape[:2]

# Exact 150 pixels left and 80 pixels up[cite: 64]
tx = -150
ty = -80

# Create translation matrix[cite: 64]
M_translate = np.float32([
    [1, 0, tx],
    [0, 1, ty]
])

centered_map = cv2.warpAffine(image, M_translate, (width, height))
Task 5: The Robot's Assembly Line (Rigid Transformation)A rigid transformation must perfectly preserve the shape, requiring a scale of exactly 1.0 while combining rotation and translation into a single matrix.   Pythonimport cv2
import numpy as np

image = cv2.imread('microchip.jpg')
height, width = image.shape[:2]
center = (width // 2, height // 2)

# Rigid transformation preserves exact shape (scale = 1.0)
M_rigid = cv2.getRotationMatrix2D(center, angle=15, scale=1.0)

# Combine Translation into the 3x3 matrix
M_rigid[0, 2] += 40 # Adjust X to center
M_rigid[1, 2] -= 20 # Adjust Y to center

aligned_chip = cv2.warpAffine(image, M_rigid, (width, height))
Task 6: The Architect's Blueprint (Similarity Transformation)A similarity transformation combines scaling, rotation, and translation into one single mathematical operation, preserving angles but changing size.   Pythonimport cv2
import numpy as np

image = cv2.imread('blueprint.jpg')
height, width = image.shape[:2]
center = (width // 2, height // 2)

# Combine Scale, Rotation, and Translation into one single step[cite: 63]
scale_factor = 2.0
angle = 90
M_sim = cv2.getRotationMatrix2D(center, angle, scale_factor)

# Fix the corner placement
M_sim[0, 2] += 200
M_sim[1, 2] -= 100

fixed_blueprint = cv2.warpAffine(image, M_sim, (int(width * scale_factor), int(height * scale_factor)))
Task 7: The Glitched Image (General Affine Transformation)To fix an image distorted by scaling, rotation, shearing, and translation simultaneously, a system of linear equations must be solved using exactly three corresponding coordinate points[cite: 63].Pythonimport cv2
import numpy as np

image = cv2.imread('glitched_photo.jpg')
height, width = image.shape[:2]

# Starting and ending coordinates of three distinct landmarks[cite: 63]
pts_src = np.float32([[50, 50], [200, 50], [50, 200]])
pts_dst = np.float32([[10, 100], [200, 50], [100, 250]])

# Solve for the 6 unknown values of the general Affine matrix[cite: 63]
M_affine = cv2.getAffineTransform(pts_src, pts_dst)

fixed_photo = cv2.warpAffine(image, M_affine, (width, height))
Task 8: The Sidewalk Illusion (Projective Perspective)To reverse perspective distortion and create a top-down view, a projective transformation maps four original corners to a perfect mathematical square[cite: 63].Pythonimport cv2
import numpy as np

image = cv2.imread('chalk_art.jpg')

# Pick 4 corners of the art[cite: 63]
# Format: [Top-Left, Top-Right, Bottom-Right, Bottom-Left]
src_points = np.float32([[200, 150], [400, 150], [550, 450], [50, 450]])

# Mathematically form a perfect square for a birds-eye view[cite: 63]
dst_points = np.float32([[0, 0], [500, 0], [500, 500], [0, 500]])

# Build perspective transformation matrix[cite: 63]
M_perspective = cv2.getPerspectiveTransform(src_points, dst_points)

birds_eye_view = cv2.warpPerspective(image, M_perspective, (500, 500))
Task 9: The Stadium Panorama (Projective Homography)A homography matrix mathematically projects the perspective of a second camera into the spatial coordinates of the first camera using four shared points.   Pythonimport cv2
import numpy as np

cam1_img = cv2.imread('stadium_left.jpg')
cam2_img = cv2.imread('stadium_right.jpg')

# Select 4 matching points found in both images
pts_cam2 = np.float32([[15, 60], [15, 210], [90, 60], [90, 210]])
pts_cam1 = np.float32([[410, 65], [410, 205], [480, 60], [480, 210]])

# Calculate the Homography matrix[cite: 62]
H = cv2.getPerspectiveTransform(pts_cam2, pts_cam1)

# Stitch them together into a seamless panorama[cite: 62]
panorama_width = cam1_img.shape[1] + cam2_img.shape[1]
stitched_pano = cv2.warpPerspective(cam2_img, H, (panorama_width, cam1_img.shape[0]))

# Overlay the first image onto the canvas
stitched_pano[0:cam1_img.shape[0], 0:cam1_img.shape[1]] = cam1_img
Task 10: The Master Forger (Full Hierarchy Challenge)This process traverses the entire hierarchy by combining a linear scale, an affine rigid transformation, and a projective warp[cite: 62].Pythonimport cv2
import numpy as np

painting = cv2.imread('painting.jpg')
frame_bg = cv2.imread('museum_wall.jpg')
h_bg, w_bg = frame_bg.shape[:2]

# 1. Linear Scale: Shrink the painting[cite: 62]
shrunk_painting = cv2.resize(painting, None, fx=0.3, fy=0.3)
h_p, w_p = shrunk_painting.shape[:2]

# 2. Rigid Transformation: Move it near the frame[cite: 62]
# Using a 2x3 translation matrix (no rotation, scale=1.0)
M_rigid = np.float32([[1, 0, 400], [0, 1, 300]])
placed_painting = cv2.warpAffine(shrunk_painting, M_rigid, (w_bg, h_bg))

# 3. Projective Transformation: Warp corners into the 3D perspective of the angled frame[cite: 62]
# Source corners (the placed painting's current position)
src_corners = np.float32([
    [400, 300], [400 + w_p, 300],
    [400 + w_p, 300 + h_p], [400, 300 + h_p]
])

# Target corners (the empty, angled frame on the wall)
dst_frame_corners = np.float32([
    [415, 280], [415 + w_p - 40, 320],
    [420 + w_p - 40, 320 + h_p - 80], [420, 280 + h_p - 40]
])

M_projective = cv2.getPerspectiveTransform(src_corners, dst_frame_corners)
final_forgery = cv2.warpPerspective(placed_painting, M_projective, (w_bg, h_bg))

# Combine the warped painting with the original background
mask = cv2.cvtColor(final_forgery, cv2.COLOR_BGR2GRAY)
_, mask = cv2.threshold(mask, 1, 255, cv2.THRESH_BINARY)
mask_inv = cv2.bitwise_not(mask)
bg = cv2.bitwise_and(frame_bg, frame_bg, mask=mask_inv)
result = cv2.add(bg, final_forgery)